In [2]:
from datetime import datetime, timedelta
import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_EUR_USD
import numpy as np
import pandas as pd



In [3]:
start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31)

raw_ticks = dukascopy_python.fetch(
    instrument=INSTRUMENT_FX_MAJORS_EUR_USD,
    interval=dukascopy_python.INTERVAL_TICK,
    offer_side=dukascopy_python.OFFER_SIDE_BID,
    start=start,
    end=end,
)

print(f"Fetched {len(raw_ticks):,} ticks")
print(f"Columns: {list(raw_ticks.columns)}")
print(f"Range: {raw_ticks.index.min()} -> {raw_ticks.index.max()}")
raw_ticks.head()

INFO:DUKASCRIPT:current timestamp :2025-01-02T10:39:00.503000
INFO:DUKASCRIPT:current timestamp :2025-01-02T15:33:12.837000
INFO:DUKASCRIPT:current timestamp :2025-01-02T18:14:34.329000
INFO:DUKASCRIPT:current timestamp :2025-01-03T09:07:14.322000
INFO:DUKASCRIPT:current timestamp :2025-01-03T16:10:17.742000
INFO:DUKASCRIPT:current timestamp :2025-01-06T03:02:52.779000
INFO:DUKASCRIPT:current timestamp :2025-01-06T12:11:22.580000
INFO:DUKASCRIPT:current timestamp :2025-01-06T14:46:41.897000
INFO:DUKASCRIPT:current timestamp :2025-01-06T16:56:29.110000
INFO:DUKASCRIPT:current timestamp :2025-01-07T04:31:09.192000
INFO:DUKASCRIPT:current timestamp :2025-01-07T12:27:15.148000
INFO:DUKASCRIPT:current timestamp :2025-01-07T16:35:03.113000
INFO:DUKASCRIPT:current timestamp :2025-01-07T21:18:33.660000
INFO:DUKASCRIPT:current timestamp :2025-01-08T10:12:59.053000
INFO:DUKASCRIPT:current timestamp :2025-01-08T14:08:05.821000
INFO:DUKASCRIPT:current timestamp :2025-01-08T17:21:40.334000
INFO:DUK

Fetched 23,409,558 ticks
Columns: ['bidPrice', 'askPrice', 'bidVolume', 'askVolume']
Range: 2025-01-01 22:00:14.647000+00:00 -> 2025-12-30 22:59:47.863000+00:00


,bidPrice,askPrice,bidVolume,askVolume
timestamp,,,,
2025-01-01 22:00:14.647000+00:00,1.03503,1.03591,4.50,4.50
2025-01-01 22:00:14.949000+00:00,1.03510,1.03585,0.72,0.72
2025-01-01 22:00:19.212000+00:00,1.03514,1.03585,0.75,0.72
2025-01-01 22:02:34.026000+00:00,1.03514,1.03590,0.75,4.50
2025-01-01 22:04:01.734000+00:00,1.03514,1.03612,0.75,0.90


In [10]:
raw_ticks['mid'] = (raw_ticks['bidPrice']+ raw_ticks['askPrice'])/2
raw_ticks['mid'].head()

timestamp
2025-01-01 22:00:14.647000+00:00    1.035470
2025-01-01 22:00:14.949000+00:00    1.035475
2025-01-01 22:00:19.212000+00:00    1.035495
2025-01-01 22:02:34.026000+00:00    1.035520
2025-01-01 22:04:01.734000+00:00    1.035630
Name: mid, dtype: float64

In [11]:
d = raw_ticks['mid'].resample('1s').last().ffill()

In [12]:
d = pd.DataFrame(d)

In [13]:
d.columns

Index(['mid'], dtype='str')

In [14]:
d['log_return'] = np.log(d['mid']/d['mid'].shift(1))
Y= np.square(d['log_return'])

In [ ]:
d['cumul_sum'] =np.cumsum(Y) #realized quadratic variation your discretized ⟨M⟩_t, evaluated at every tick. This is the volatility odometer reading after each tick.


In [19]:
Z = d['cumul_sum']

In [67]:
total_qv = Z.iloc[-1]
delta = total_qv /400

In [68]:
print(delta)

1.7460424881165406e-05


In [69]:
bar_id = np.floor(Z/ delta)
new_bar = bar_id.diff().fillna(0) > 0

In [70]:
M = d['log_return'].cumsum() # M_t: cumulative return

In [71]:
Bt = M[new_bar]

In [72]:
Bt = pd.concat([pd.Series([0.0]), Bt]).reset_index(drop=True)

In [73]:
M.iloc[0] = 0

In [74]:
M.iloc[0]

np.float64(0.0)

In [75]:
"""
Validate whether B_t (M sampled at the variance-clock stopping times T_t)
behaves like standard Brownian motion.

Assumes you already have `Bt`: a pandas Series of B_t values, indexed
0, 1, 2, ... (bar number), e.g. from:

    Bt = M[new_bar]
    Bt = pd.concat([pd.Series([0.0]), Bt]).reset_index(drop=True)
"""

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

# ---- 0. Increments of B_t -------------------------------------------------
dBt = Bt.diff().dropna()

print("Increment stats:")
print(dBt.describe())
print(f"mean={dBt.mean():.6e}  var={dBt.var():.6e}\n")

# ---- 1. QQ-plot: are increments ~ Gaussian at bar-Delta scale? -----------
fig, ax = plt.subplots(figsize=(5, 5))
stats.probplot(dBt, dist="norm", plot=ax)
ax.set_title("QQ-plot of B_t increments vs Normal")
plt.tight_layout()
plt.savefig("qqplot_dBt.png", dpi=150)
plt.close()

jb_stat, jb_p = stats.jarque_bera(dBt)
print(f"Jarque-Bera normality test: stat={jb_stat:.3f}, p={jb_p:.4f}")
print("  (large samples make JB oversensitive to tiny deviations -- "
      "eyeball the QQ-plot too, don't rely on the p-value alone)\n")

# ---- 2. ACF of raw increments: should already be ~0 at all lags ---------
# (true Brownian increments are iid, so this should hold even before the
# time-change -- it's the squared-increment test below that's diagnostic)
fig, ax = plt.subplots(figsize=(8, 4))
sm.graphics.tsa.plot_acf(dBt, lags=40, ax=ax)
ax.set_title("ACF of B_t increments")
plt.tight_layout()
plt.savefig("acf_dBt.png", dpi=150)
plt.close()

lb_returns = acorr_ljungbox(dBt, lags=[10, 20, 40], return_df=True)
print("Ljung-Box on raw increments (want: p-values NOT small):")
print(lb_returns, "\n")

# ---- 3. ACF of SQUARED increments: THE real test for volatility clustering
sq_dBt = dBt**2

fig, ax = plt.subplots(figsize=(8, 4))
sm.graphics.tsa.plot_acf(sq_dBt, lags=40, ax=ax)
ax.set_title("ACF of squared B_t increments")
plt.tight_layout()
plt.savefig("acf_sq_dBt.png", dpi=150)
plt.close()

lb_sq = acorr_ljungbox(sq_dBt, lags=[10, 20, 40], return_df=True)
print("Ljung-Box on squared increments (want: p-values NOT small):")
print(lb_sq, "\n")

# ---- 4. Optional: side-by-side comparison against naive calendar-time ----
# Run the same three checks (QQ, ACF, Ljung-Box on squares) on a fixed
# calendar-interval return series (e.g. your onesec / 1-minute returns).
# If the time-change is doing its job, the calendar-time squared returns
# should show strong, significant autocorrelation (small p-values --
# classic volatility clustering) while the B_t squared increments above
# should not. That contrast is your evidence the Dubins-Schwarz time-change
# is working on this data.

print("Interpretation:")
print("- QQ-plot hugging the diagonal + high JB p-value -> increments look Gaussian")
print("- Ljung-Box on raw increments: not very informative on its own")
print("  (FX returns are usually already ~uncorrelated in calendar time)")
print("- Ljung-Box on SQUARED increments is the key test:")
print("  small p-values -> still volatility clustering, Delta may be too coarse")
print("  or QV estimate too noisy (revisit microstructure-noise handling)")
print("  large p-values -> clustering absorbed by the variance clock, as expected")

Increment stats:
count    389.000000
mean       0.000330
std        0.004083
min       -0.016142
25%       -0.002410
50%        0.000162
75%        0.002599
max        0.011981
dtype: float64
mean=3.301687e-04  var=1.666892e-05

Jarque-Bera normality test: stat=2.481, p=0.2892
  (large samples make JB oversensitive to tiny deviations -- eyeball the QQ-plot too, don't rely on the p-value alone)

Ljung-Box on raw increments (want: p-values NOT small):
      lb_stat  lb_pvalue
10   2.471701   0.991282
20   9.619174   0.974564
40  20.941667   0.994374 

Ljung-Box on squared increments (want: p-values NOT small):
      lb_stat  lb_pvalue
10   5.010918   0.890447
20  16.239741   0.701644
40  47.420002   0.195793 

Interpretation:
- QQ-plot hugging the diagonal + high JB p-value -> increments look Gaussian
- Ljung-Box on raw increments: not very informative on its own
  (FX returns are usually already ~uncorrelated in calendar time)
- Ljung-Box on SQUARED increments is the key test:
  small p

In [76]:
"""
Sweep over candidate bar counts / delta values and compare how
"Brownian-like" the resulting B_t looks for each.

Requires: qv (the cumulative QV / odometer series, e.g. your 'cumul_sum'
column) and M (cumulative signed-return series, log_return.cumsum()),
both indexed the same way.
"""

import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
qv = Z

def build_Bt(qv: pd.Series, M: pd.Series, num_bars: int) -> pd.Series:
    total_qv = qv.iloc[-1]
    delta = total_qv / num_bars

    bar_id = np.floor(qv / delta)
    new_bar = bar_id.diff().fillna(0) > 0

    Bt = M[new_bar]
    Bt = pd.concat([pd.Series([0.0]), Bt]).reset_index(drop=True)
    return Bt, delta


def evaluate_delta(qv: pd.Series, M: pd.Series, num_bars: int, lags=(10, 20, 40)) -> dict:
    Bt, delta = build_Bt(qv, M, num_bars)
    dBt = Bt.diff().dropna()

    if len(dBt) < max(lags) + 5:
        # not enough bars to run the diagnostics meaningfully
        return {
            "num_bars": num_bars, "delta": delta, "n_increments": len(dBt),
            "jb_p": np.nan, "lb_raw_p_max_lag": np.nan, "lb_sq_p_max_lag": np.nan,
        }

    jb_stat, jb_p = stats.jarque_bera(dBt)
    lb_raw = acorr_ljungbox(dBt, lags=list(lags), return_df=True)
    lb_sq = acorr_ljungbox(dBt**2, lags=list(lags), return_df=True)

    return {
        "num_bars": num_bars,
        "delta": delta,
        "n_increments": len(dBt),
        "jb_p": jb_p,                                   # want: high
        "lb_raw_p_max_lag": lb_raw["lb_pvalue"].iloc[-1],  # want: high
        "lb_sq_p_max_lag": lb_sq["lb_pvalue"].iloc[-1],    # want: high (key test)
    }


# ---- sweep -----------------------------------------------------------
candidate_num_bars = [50, 100, 150, 225, 400, 800]

results = pd.DataFrame([
    evaluate_delta(qv, M, n) for n in candidate_num_bars
])

print(results.to_string(index=False))

# ---- how to read this table -------------------------------------------
# n_increments      -- sanity check: too few (<~50-100) and the p-values
#                       above aren't trustworthy regardless of their value
# jb_p              -- higher = increments look more Gaussian
# lb_raw_p_max_lag  -- higher = no leftover autocorrelation in raw increments
#                       (usually fine across the whole grid for FX)
# lb_sq_p_max_lag   -- higher = no leftover volatility clustering
#                       (THE diagnostic that should respond to delta choice --
#                       watch for it dropping as num_bars grows past some point,
#                       or staying low throughout if delta is never fine enough)
#
# Look for the region where lb_sq_p_max_lag is consistently high AND
# n_increments is large enough to trust that number. If it's high nowhere,
# the microstructure-noise handling (step 4) likely needs revisiting before
# delta-tuning alone will fix it.

 num_bars    delta  n_increments     jb_p  lb_raw_p_max_lag  lb_sq_p_max_lag
       50 0.000140            49 0.047997          0.461160         0.920225
      100 0.000070            98 0.031189          0.954825         0.000148
      150 0.000047           148 0.208675          0.299851         0.003855
      225 0.000031           222 0.586579          0.988196         0.046859
      400 0.000017           389 0.289231          0.994374         0.195793
      800 0.000009           771 0.017018          0.901262         0.856320


In [78]:
"""
Validate whether B_t (M sampled at the variance-clock stopping times T_t)
behaves like standard Brownian motion.

Assumes you already have `Bt`: a pandas Series of B_t values, indexed
0, 1, 2, ... (bar number), e.g. from:

    Bt = M[new_bar]
    Bt = pd.concat([pd.Series([0.0]), Bt]).reset_index(drop=True)
"""

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

# ---- 0. Increments of B_t -------------------------------------------------
dBt = Bt.diff().dropna()

print("Increment stats:")
print(dBt.describe())
print(f"mean={dBt.mean():.6e}  var={dBt.var():.6e}\n")

# ---- 1. QQ-plot: are increments ~ Gaussian at bar-Delta scale? -----------
fig, ax = plt.subplots(figsize=(5, 5))
stats.probplot(dBt, dist="norm", plot=ax)
ax.set_title("QQ-plot of B_t increments vs Normal")
plt.tight_layout()
plt.savefig("qqplot_dBt.png", dpi=150)
plt.close()

jb_stat, jb_p = stats.jarque_bera(dBt)
print(f"Jarque-Bera normality test: stat={jb_stat:.3f}, p={jb_p:.4f}")
print("  (large samples make JB oversensitive to tiny deviations -- "
      "eyeball the QQ-plot too, don't rely on the p-value alone)\n")

# ---- 1b. Histogram + fitted Normal curve + KDE curve ----------------------
# Gives you the "shape" intuition the QQ scatter doesn't: does the empirical
# distribution actually look bell-shaped, and where does it deviate?
mu, sigma = dBt.mean(), dBt.std()
x = np.linspace(dBt.min(), dBt.max(), 300)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(dBt, bins=30, density=True, alpha=0.45, color="steelblue",
        label="Empirical B_t increments")
ax.plot(x, stats.norm.pdf(x, mu, sigma), "r-", lw=2,
        label=f"Normal fit (mu={mu:.2e}, sigma={sigma:.2e})")

kde = stats.gaussian_kde(dBt)
ax.plot(x, kde(x), "g--", lw=2, label="KDE of empirical data")

ax.set_title("Distribution of B_t increments vs Normal curve")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("hist_normal_dBt.png", dpi=150)
plt.close()

# ---- 1c. CDF comparison: empirical vs Normal, both as smooth curves ------
# Same information as the QQ-plot, but rendered as two overlapping S-curves
# instead of a scatter -- easier to see WHERE (which range of values) the
# empirical distribution departs from Normal (tails vs center).
x_sorted = np.sort(dBt.values)
empirical_cdf = np.arange(1, len(x_sorted) + 1) / len(x_sorted)
theoretical_cdf = stats.norm.cdf(x_sorted, mu, sigma)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_sorted, empirical_cdf, color="steelblue", lw=2, label="Empirical CDF")
ax.plot(x_sorted, theoretical_cdf, color="red", lw=2, ls="--", label="Normal CDF")
ax.set_title("Empirical vs Normal CDF of B_t increments")
ax.set_xlabel("increment value")
ax.set_ylabel("cumulative probability")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("cdf_compare_dBt.png", dpi=150)
plt.close()

# ---- 2. ACF of raw increments: should already be ~0 at all lags ---------
# (true Brownian increments are iid, so this should hold even before the
# time-change -- it's the squared-increment test below that's diagnostic)
fig, ax = plt.subplots(figsize=(8, 4))
sm.graphics.tsa.plot_acf(dBt, lags=40, ax=ax)
ax.set_title("ACF of B_t increments")
plt.tight_layout()
plt.savefig("acf_dBt.png", dpi=150)
plt.close()

lb_returns = acorr_ljungbox(dBt, lags=[10, 20, 40], return_df=True)
print("Ljung-Box on raw increments (want: p-values NOT small):")
print(lb_returns, "\n")

# ---- 3. ACF of SQUARED increments: THE real test for volatility clustering
sq_dBt = dBt**2

fig, ax = plt.subplots(figsize=(8, 4))
sm.graphics.tsa.plot_acf(sq_dBt, lags=40, ax=ax)
ax.set_title("ACF of squared B_t increments")
plt.tight_layout()
plt.savefig("acf_sq_dBt.png", dpi=150)
plt.close()

lb_sq = acorr_ljungbox(sq_dBt, lags=[10, 20, 40], return_df=True)
print("Ljung-Box on squared increments (want: p-values NOT small):")
print(lb_sq, "\n")

# ---- 4. Optional: side-by-side comparison against naive calendar-time ----
# Run the same three checks (QQ, ACF, Ljung-Box on squares) on a fixed
# calendar-interval return series (e.g. your onesec / 1-minute returns).
# If the time-change is doing its job, the calendar-time squared returns
# should show strong, significant autocorrelation (small p-values --
# classic volatility clustering) while the B_t squared increments above
# should not. That contrast is your evidence the Dubins-Schwarz time-change
# is working on this data.

print("Interpretation:")
print("- QQ-plot hugging the diagonal + high JB p-value -> increments look Gaussian")
print("- Ljung-Box on raw increments: not very informative on its own")
print("  (FX returns are usually already ~uncorrelated in calendar time)")
print("- Ljung-Box on SQUARED increments is the key test:")
print("  small p-values -> still volatility clustering, Delta may be too coarse")
print("  or QV estimate too noisy (revisit microstructure-noise handling)")
print("  large p-values -> clustering absorbed by the variance clock, as expected")

Increment stats:
count    389.000000
mean       0.000330
std        0.004083
min       -0.016142
25%       -0.002410
50%        0.000162
75%        0.002599
max        0.011981
dtype: float64
mean=3.301687e-04  var=1.666892e-05

Jarque-Bera normality test: stat=2.481, p=0.2892
  (large samples make JB oversensitive to tiny deviations -- eyeball the QQ-plot too, don't rely on the p-value alone)

Ljung-Box on raw increments (want: p-values NOT small):
      lb_stat  lb_pvalue
10   2.471701   0.991282
20   9.619174   0.974564
40  20.941667   0.994374 

Ljung-Box on squared increments (want: p-values NOT small):
      lb_stat  lb_pvalue
10   5.010918   0.890447
20  16.239741   0.701644
40  47.420002   0.195793 

Interpretation:
- QQ-plot hugging the diagonal + high JB p-value -> increments look Gaussian
- Ljung-Box on raw increments: not very informative on its own
  (FX returns are usually already ~uncorrelated in calendar time)
- Ljung-Box on SQUARED increments is the key test:
  small p